# Tone Aligner — Single-Run DPO Pipeline (Colab)

Click **Runtime -> Run all** to execute the entire pipeline in one go:
1. Sync repo & install dependencies
2. Build balanced preference dataset (`PKU-SafeRLHF` 50% + `UltraFeedback` 50%)
3. Train 4-bit QLoRA DPO on `alignment-handbook/zephyr-7b-sft-full` (1× L4 or A100)
4. Evaluate Base (SFT) vs. DPO-Aligned on 100 prompts (Toxicity, Reward Score/Win-Rate, Safety & Over-Refusal)
5. Print the final copy-pasteable report, display charts, and download `tone_align_outputs.zip`

> **Reminder:** Push your latest local changes to GitHub before running so Colab pulls the latest `config.py` and scripts.


In [ ]:
# ============================================================
# ONE-CLICK PIPELINE: Setup -> Data -> Train -> Evaluate
# ============================================================
import os
import subprocess
import sys
import torch

# --- Settings ---
RUN_MODE = 'full'       # 'full' (4,800 train, ~1.8-2h on L4 / ~40m on A100) or 'sanity' (1,600 train)
EVAL_EXTRA_PKU = 40     # Extra adversarial test prompts from PKU-SafeRLHF (total = 60 + 40 = 100)
EVAL_BATCH_SIZE = 8
REPO_URL = 'https://github.com/manohar135/LLM-Tone-Alignment-DPO.git'
REPO_DIR = '/content/LLM-Tone-Alignment-DPO'


def run_step(title, cmd):
    print(f'\n{"=" * 72}\n  {title}\n{"=" * 72}', flush=True)
    subprocess.run(cmd, check=True)


# 1. Check GPU
assert torch.cuda.is_available(), 'No GPU detected! Switch Colab runtime to NVIDIA L4 or A100.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | bf16: {torch.cuda.is_bf16_supported()}', flush=True)

# 2. Clone or update repo
if not os.path.exists(REPO_DIR):
    run_step('STEP 1/5: Cloning Repository', ['git', 'clone', REPO_URL, REPO_DIR])
else:
    run_step('STEP 1/5: Pulling Latest Repo Changes', ['git', '-C', REPO_DIR, 'pull'])

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}', flush=True)

# 3. Install dependencies
run_step('STEP 2/5: Installing Dependencies', [sys.executable, '-m', 'pip', 'install', '-q', '-U', '-r', 'requirements.txt'])

# 4. Prepare dataset
sanity_flag = ['--sanity'] if RUN_MODE == 'sanity' else []
run_step(f'STEP 3/5: Preparing Dataset ({RUN_MODE})', [sys.executable, 'data_preparation.py', *sanity_flag])

# 5. Train DPO
run_step(f'STEP 4/5: Training DPO ({RUN_MODE})', [sys.executable, 'train_dpo.py', *sanity_flag])

# 6. Evaluate Base (SFT) vs. DPO-Aligned
run_step(
    'STEP 5/5: Running Multi-Metric Evaluation',
    [sys.executable, 'evaluate.py', '--batch-size', str(EVAL_BATCH_SIZE), '--hf-safety-n', str(EVAL_EXTRA_PKU)],
)

print('\nALL PIPELINE STEPS COMPLETED SUCCESSFULLY!')


In [ ]:
# ============================================================
# FINAL REPORT, CHARTS & OUTPUT ZIP DOWNLOAD
# ============================================================
import json
import os
import subprocess
from IPython.display import Image, display

tm = json.load(open('outputs/training_metrics.json', encoding='utf-8'))
ev = json.load(open('outputs/eval_results.json', encoding='utf-8'))
fe = tm.get('final_eval_metrics', {})
s = ev['summary']

print('=' * 76)
print('  FINAL COPY-PASTE REPORT')
print('=' * 76)
print(f"Model:                  {tm['model_id']}")
print(f"GPU:                    {tm['gpu']} ({tm['vram_gb']} GB)")
print(f"Samples (Train / Eval): {tm['train_samples']} / {tm['eval_samples']}")
print(f"Trainable Params:       {tm['trainable_params'] / 1e6:.2f}M ({tm['trainable_pct']:.2f}%)")
print(f"Train Wall Time:        {tm['train_minutes']:.1f} min")
print(f"Final Eval Loss:        {fe.get('eval_loss', float('nan')):.4f}")
print(f"Eval Preference Acc:    {fe.get('eval_rewards/accuracies', 0.0) * 100:.1f}%")
print(f"Eval Reward Margin:     {fe.get('eval_rewards/margins', 0.0):+.4f}")
print('-' * 76)
print(f"Eval Prompts:           {s['num_prompts']}")
print(f"Avg Reward Score:       Base {s['avg_base_reward']:+.4f} -> Aligned {s['avg_aligned_reward']:+.4f} (Delta: {s['reward_delta']:+.4f})")
print(f"Reward Win-Rate:        {s['reward_win_rate_pct']:.1f}% Win | {s['reward_tie_rate_pct']:.1f}% Tie | {s['reward_loss_rate_pct']:.1f}% Loss")
print(f"Avg Toxicity:           Base {s['avg_base_toxicity']:.4f} -> Aligned {s['avg_aligned_toxicity']:.4f}")
print(f"Safety Refusal (Harm):  Base {s['safety_refusal_rate_base_pct']:.1f}% -> Aligned {s['safety_refusal_rate_aligned_pct']:.1f}%")
print(f"False Refusal (Benign): Base {s['false_refusal_rate_base_pct']:.1f}% -> Aligned {s['false_refusal_rate_aligned_pct']:.1f}%")
print('-' * 76)
print(f"{'Category':<16} {'N':>3} {'Base RM':>8} {'Algn RM':>8} {'Win%':>6} {'Base Tox':>9} {'Algn Tox':>9} {'Ref B->A':>11}")
for cat, c in s['per_category'].items():
    ref = f"{c['base_refusal_rate']:.0f}%->{c['aligned_refusal_rate']:.0f}%"
    print(f"{cat:<16} {c['count']:>3} {c['base_reward']:>+8.3f} {c['aligned_reward']:>+8.3f} {c['reward_win_rate']:>5.1f}% {c['base_toxicity']:>9.4f} {c['aligned_toxicity']:>9.4f} {ref:>11}")
print('=' * 76)

# Display charts inline
for img_path in ['outputs/training_curves.png', 'outputs/eval_comparison.png']:
    if os.path.exists(img_path):
        display(Image(img_path))

# Package adapter, metrics, and charts for download (excludes intermediate checkpoints)
subprocess.run([
    'zip', '-q', '-r', 'tone_align_outputs.zip',
    'outputs/dpo_adapter',
    'outputs/training_metrics.json',
    'outputs/eval_results.json',
    'outputs/training_curves.png',
    'outputs/eval_comparison.png',
    '-x', 'outputs/dpo_adapter/checkpoint-*/*',
], check=True)
print('\nPackaged outputs to tone_align_outputs.zip')

try:
    from google.colab import files
    files.download('tone_align_outputs.zip')
except ImportError:
    pass
